In [ ]:
# ---- overnight sweep of Tasks 2 / 4 / 5 / 6: what runs and until when ------------------------------------------------
# every sweep has a time budget (minutes): it starts no new run once its budget is used up or the overall end is reached
# (a run that has started always finishes). The grids run in random order -> a sweep cut by its budget still covers its range.
END_TIME = "18:00"                              # overall end (today, clock of this PC); "end_at" in overnight_queue.json can move it
run_task2 = True                                # item cosine, EASE, Mult-VAE                          budgets ~1.4 h
run_task4 = True                                # item cosine, EASE, time-weighted EASE, Mult-VAE      ~0.9 h
run_task5 = True                                # transition, item cosine, EASE, Mult-VAE, SASRec      ~2.4 h
run_task6 = True                                # LightGBM gap model                                   ~0.5 h
run_queue = True                                # then the refinement sweeps in overnight_queue.json, until the overall end

# Overnight sweep: Tasks 2, 4, 5, 6

**Run All** once: the notebook runs every sweep below one after the other and stops at `END_TIME`.
Every sweep cell only runs if the flag of its task is `True`.

**Order:** fast models first (many runs, wide grids), then the minute-long EASE fits, then the GPU networks
(few runs, only the parameters that mattered most so far):

| phase | models | time per run (RTX 5080 PC, measured / estimated) |
|---|---|---|
| 1 fast | Task 6 LightGBM, Task 5 transition, item cosine (Tasks 2 / 4 / 5), Task 4 EASE + time-weighted EASE | 1 s - 1 min |
| 2 EASE in RAM | Task 2 EASE, Task 5 EASE + time-weighted EASE | ~3 min (CPU, ~21 GB RAM) |
| 3 GPU networks | Mult-VAE (Tasks 2 / 4 / 5), SASRec (Task 5) | ~2 - 6 min (estimate) |
| 4 refinements | the sweeps in `overnight_queue.json` (written while the notebook runs) | |

**Time budgets:** each sweep stops starting new runs after its budget (`budget_minutes`) or at `END_TIME`; the run in progress
finishes. The combinations run in random order, so a sweep that is cut still samples its whole grid.
A staged sweep passes the best values of its earlier stages on (`best_values_of`).

**Nothing stops the night:** a failing sweep is printed and skipped (its finished runs stay in its output file), the next one starts.
The computer is kept awake until the last cell.

**Phase 4 (refinement queue):** after phase 3 the notebook reads `sweep/overnight_queue.json` every minute and runs every entry it
has not run yet, e.g. a finer grid around a best value of the night. Entries can be added while the notebook runs; `"end_at"`
(`"YYYY-MM-DD HH:MM"`) moves the overall end. Entry format:
`{"name": "...", "task": "task5", "model": "transition", "grid": {...}, "budget_minutes": 20, "options": {"rerun_finalists_on_other_splits": true}}`

**Results:** one file per sweep in `output/<task>/<model>_<start time>.json` (as in the `taskN_sweep` notebooks: `best.parameters` can be
copied into `lib/config.py`), one line per sweep in `output/overnight_log.txt`, and a summary table in the last cell.
Scores: pair accuracy (Tasks 2 / 4 / 5, higher is better), MAE in days (Task 6, lower is better).
No refit, nothing is written to `trained_models/`.

**No autoreload here:** the night runs with the code and `lib/config.py` as they were when the notebook started.

In [ ]:
import json
import time
import traceback
from datetime import datetime, timedelta
from pathlib import Path

import torch

from grid_search import OUTPUT_DIR, SWEEP_DIR, best_values_of, keep_computer_awake, print_sweep_overview, run_grid_search

print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

QUEUE_PATH = SWEEP_DIR / "overnight_queue.json"
LOG_PATH = OUTPUT_DIR / "overnight_log.txt"
NO_RESULT = {"best": None}                      # what a skipped or failed sweep returns (best_values_of -> {})

overnight_start = datetime.now()
end_hour, end_minute = map(int, END_TIME.split(":"))
overall_end = overnight_start.replace(hour=end_hour, minute=end_minute, second=0, microsecond=0)
if overall_end <= overnight_start:              # END_TIME already passed today -> tomorrow
    overall_end += timedelta(days=1)
overnight_results = {}                          # sweep label -> report (summary in the last cell)


def write_log(line: str):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with open(LOG_PATH, "a", encoding="utf-8") as log_file:
        log_file.write(f"{datetime.now():%Y-%m-%d %H:%M:%S} {line}\n")


# one sweep with a time budget: no new run after budget_minutes or at the overall end, combinations in random order
# (a cut sweep still covers the whole grid); a failing sweep is printed and skipped, the night goes on
def run_timeboxed_sweep(label: str, task_name: str, model_name: str, parameter_grid: dict, budget_minutes: float, **options) -> dict:
    now = datetime.now()
    if now >= overall_end - timedelta(minutes=1):
        print(f"\nskipped {label}: overall end {overall_end:%H:%M} reached", flush=True)
        write_log(f"skipped  {label} (overall end reached)")
        return NO_RESULT
    deadline = min(now + timedelta(minutes=budget_minutes), overall_end)
    print(f"\n===== {label} | budget {budget_minutes:.0f} min -> no new run after {deadline:%H:%M} =====", flush=True)
    write_log(f"start    {label} | {task_name} / {model_name} | until {deadline:%H:%M}")
    try:
        report = run_grid_search(task_name, model_name, parameter_grid, deadline=deadline, random_order_seed=0, **options)
    except Exception as error:
        traceback.print_exc()
        print(f"{label} failed -> continuing with the next sweep", flush=True)
        write_log(f"FAILED   {label}: {type(error).__name__}: {error}"[:400])
        return NO_RESULT
    finally:
        keep_computer_awake(True)               # run_grid_search lets the computer sleep again at its end -> hold it all night

    overnight_results[label] = report
    output_name = f"{task_name}/{model_name}_{report['started'].replace(' ', '_').replace(':', '-')}.json"
    best = report["best"]
    write_log(f"finished {label} | {report['status']} | {report['runs_finished']} runs | "
              + (f"best {best['score']:.5f} with {json.dumps(best['swept_values'])}" if best else "no run finished")
              + f" | {output_name}")
    return report


keep_computer_awake(True)
write_log(f"overnight sweep started, overall end {overall_end:%Y-%m-%d %H:%M}")
print(f"start {overnight_start:%H:%M} | overall end {overall_end:%Y-%m-%d %H:%M} | log: {LOG_PATH}")

## Phase 1 - fast models: wide grids
### Task 6: LightGBM gap model (CPU, ~5 - 60 s per run)
Score: out-of-fold MAE in days on every user's last gap (lower is better). Current config: 31 leaves, min. 100 rows per leaf,
600 trees at learning rate 0.03.

In [ ]:
if run_task6:
    # stage 1 - tree shape: leaves x minimum rows per leaf (30 runs)
    lightgbm_gap_shape_sweep = run_timeboxed_sweep("T6 LightGBM 1 tree shape", "task6", "lightgbm_gap_model", {
        "lightgbm_num_leaves": [7, 15, 31, 63, 127],
        "lightgbm_min_child_samples": [10, 25, 50, 100, 200, 400],
    }, budget_minutes=8)
    lightgbm_gap_best_values = best_values_of(lightgbm_gap_shape_sweep)

    # stage 2 - learning rate x number of trees at the best shape (25 runs; 4800 trees ~8x as long as 600)
    lightgbm_gap_boosting_sweep = run_timeboxed_sweep("T6 LightGBM 2 boosting", "task6", "lightgbm_gap_model", {
        **lightgbm_gap_best_values,
        "lightgbm_learning_rate": [0.01, 0.02, 0.03, 0.05, 0.1],
        "lightgbm_n_estimators": [300, 600, 1200, 2400, 4800],
    }, budget_minutes=12)
    lightgbm_gap_best_values |= best_values_of(lightgbm_gap_boosting_sweep)

    # stage 3 - tree shape again at the best learning rate / trees (shape and boosting interact) (12 runs)
    lightgbm_gap_shape_check = run_timeboxed_sweep("T6 LightGBM 3 shape again", "task6", "lightgbm_gap_model", {
        **lightgbm_gap_best_values,
        "lightgbm_num_leaves": [15, 31, 63, 127],
        "lightgbm_min_child_samples": [25, 100, 400],
    }, budget_minutes=6)
    lightgbm_gap_best_values |= best_values_of(lightgbm_gap_shape_check)

    # stage 4 - noise check: the best settings with other fold assignments (5 runs) -> how much the MAE moves by chance
    lightgbm_gap_seed_check = run_timeboxed_sweep("T6 LightGBM 4 seed check", "task6", "lightgbm_gap_model", {
        **lightgbm_gap_best_values,
        "seed": [0, 1, 2, 3, 4],
    }, budget_minutes=3)

### Task 5: transition model (CPU, ~1 - 15 s per run)
Quick check (CPU container): config (3 steps, last 5 books) 0.702; 20 steps / last 50 books 0.732 -> much longer ranges.

In [ ]:
if run_task5:
    # stage 1 - which transitions count: steps ahead x their decay x shrinkage, at a longer context (48 runs)
    transition_counts_sweep = run_timeboxed_sweep("T5 transition 1 counts", "task5", "transition", {
        "steps_ahead": [3, 8, 20, 50],
        "step_decay": [0.5, 0.7, 0.85, 1.0],
        "shrinkage": [1.0, 5.0, 20.0],
        "context_length": 20,
        "context_decay": 0.8,
    }, budget_minutes=10)
    transition_best_values = best_values_of(transition_counts_sweep)

    # stage 2 - which of the user's last books vote: context length x decay (20 runs + the best 3 on split seeds 1 / 2)
    transition_context_sweep = run_timeboxed_sweep("T5 transition 2 context", "task5", "transition", {
        **transition_best_values,
        "context_length": [5, 10, 20, 50, 100],
        "context_decay": [0.5, 0.7, 0.85, 1.0],
    }, budget_minutes=8, rerun_finalists_on_other_splits=True, n_finalists=3)

### Item cosine similarity, Tasks 2 / 4 / 5 (CPU, ~10 - 60 s per run)
Quick checks (CPU container): more neighbours, more books per active user and less shrinkage help everywhere:

| task | config (100 neighbours, 300 books per user, shrinkage 20) | larger setting |
|---|---|---|
| 2 | 0.833 (today's sweep best: 0.841, shrinkage 10) | 400 / 1000 / 5: 0.851 |
| 4 | 0.677 | 800 / 2000 / 0: 0.699 |
| 5 | 0.723 | 400 / 1000 / 5: 0.745 |

Same grid for all three: 48 runs + the best 3 on validation split seeds 1 / 2.

In [ ]:
ITEM_COSINE_GRID = {
    "n_neighbors": [200, 400, 800, 1600],
    "max_interactions_per_user_for_similarity": [1000, 2000, 5000],
    "similarity_shrinkage": [0.0, 2.0, 5.0, 10.0],
}

if run_task2:
    task2_item_cosine_sweep = run_timeboxed_sweep("T2 item cosine", "task2", "item_cosine", ITEM_COSINE_GRID,
                                                  budget_minutes=25, rerun_finalists_on_other_splits=True, n_finalists=3)
if run_task4:
    task4_item_cosine_sweep = run_timeboxed_sweep("T4 item cosine", "task4", "item_cosine", ITEM_COSINE_GRID,
                                                  budget_minutes=12, rerun_finalists_on_other_splits=True, n_finalists=3)
if run_task5:
    task5_item_cosine_sweep = run_timeboxed_sweep("T5 item cosine", "task5", "item_cosine", ITEM_COSINE_GRID,
                                                  budget_minutes=25, rerun_finalists_on_other_splits=True, n_finalists=3)

### Task 4: EASE and time-weighted EASE (GPU, ~17k books: seconds per run)
`ease` and `time_ease` have separate configs (`TASK4_EASE` / `TASK4_TIME_EASE`).

In [ ]:
if run_task4:
    # plain EASE: regularization only (7 runs + the best 3 on split seeds 1 / 2)
    task4_ease_sweep = run_timeboxed_sweep("T4 EASE", "task4", "ease", {
        "regularization": [100.0, 250.0, 500.0, 1000.0, 2000.0, 4000.0, 8000.0],
    }, budget_minutes=4, rerun_finalists_on_other_splits=True, n_finalists=3)

    # time-weighted EASE, stage 1 - regularization x how fast older books lose weight (25 runs)
    task4_time_ease_sweep = run_timeboxed_sweep("T4 time EASE 1 decay", "task4", "time_ease", {
        "regularization": [250.0, 500.0, 1000.0, 2000.0, 4000.0],
        "history_time_decay_days": [30.0, 90.0, 180.0, 365.0, 730.0],
    }, budget_minutes=8)
    task4_time_ease_best_values = best_values_of(task4_time_ease_sweep)

    # stage 2 - the weight that even very old books keep (5 runs)
    task4_time_ease_floor_sweep = run_timeboxed_sweep("T4 time EASE 2 floor", "task4", "time_ease", {
        **task4_time_ease_best_values,
        "history_time_decay_floor": [0.0, 0.05, 0.1, 0.2, 0.4],
    }, budget_minutes=3)

## Phase 2 - EASE in RAM (CPU, ~3 min and ~21 GB RAM per run)
Task 2, today: regularization 250 / 500 / 1000 -> 0.842 / 0.849 / 0.851, still rising -> larger values.
Task 5: one config for both variants (`TASK5_EASE`): stage 1 scores plain EASE, stage 2 time-weighted EASE.

In [ ]:
if run_task2:
    # regularization above today's best 1000 (4 runs)
    task2_ease_sweep = run_timeboxed_sweep("T2 EASE", "task2", "ease", {
        "regularization": [2000.0, 4000.0, 8000.0, 16000.0],
    }, budget_minutes=13)

if run_task5:
    # stage 1 - regularization, scored as plain EASE (5 runs)
    task5_ease_sweep = run_timeboxed_sweep("T5 EASE 1 regularization", "task5", "ease", {
        "regularization": [500.0, 1000.0, 2000.0, 4000.0, 8000.0],
    }, budget_minutes=18)
    task5_ease_best_values = best_values_of(task5_ease_sweep)

    # stage 2 - time decay of the user's history at the best regularization, scored as time-weighted EASE (4 runs)
    task5_time_ease_sweep = run_timeboxed_sweep("T5 time EASE 2 decay", "task5", "time_ease", {
        **task5_ease_best_values,
        "history_time_decay_days": [60.0, 180.0, 365.0, 730.0],
    }, budget_minutes=15)

## Phase 3 - GPU networks: few runs, the parameters that mattered most
Task 2 Mult-VAE, today: input dropout 0.3 and KL weight 0.3 best (0.858), both at the edge of the grid -> further out.

In [ ]:
if run_task2:
    # stage 1 - denoising dropout x largest KL weight beta (9 runs, ~2.5 min each)
    task2_mult_vae_sweep = run_timeboxed_sweep("T2 Mult-VAE 1 dropout / KL", "task2", "mult_vae", {
        "input_dropout": [0.1, 0.2, 0.3],
        "kl_weight_max": [0.3, 0.5, 1.0],
    }, budget_minutes=25)
    task2_mult_vae_best_values = best_values_of(task2_mult_vae_sweep)

    # stage 2 - capacity: hidden x latent units (4 runs)
    task2_mult_vae_capacity_sweep = run_timeboxed_sweep("T2 Mult-VAE 2 capacity", "task2", "mult_vae", {
        **task2_mult_vae_best_values,
        "hidden_dimension": [600, 1000],
        "latent_dimension": [200, 400],
    }, budget_minutes=12)
    task2_mult_vae_best_values |= best_values_of(task2_mult_vae_capacity_sweep)

    # stage 3 - learning rate x KL annealing length (4 runs; config: 1e-3, 20 epochs)
    task2_mult_vae_optimisation_sweep = run_timeboxed_sweep("T2 Mult-VAE 3 learning rate / annealing", "task2", "mult_vae", {
        **task2_mult_vae_best_values,
        "learning_rate": [5e-4, 2e-3],
        "kl_annealing_epochs": [10, 40],
    }, budget_minutes=10)

if run_task4:
    # stage 1 - dropout x KL weight (9 runs, fewer books than Task 2 -> faster)
    task4_mult_vae_sweep = run_timeboxed_sweep("T4 Mult-VAE 1 dropout / KL", "task4", "mult_vae", {
        "input_dropout": [0.1, 0.3, 0.5],
        "kl_weight_max": [0.2, 0.5, 1.0],
    }, budget_minutes=18)
    task4_mult_vae_best_values = best_values_of(task4_mult_vae_sweep)

    # stage 2 - capacity (4 runs)
    task4_mult_vae_capacity_sweep = run_timeboxed_sweep("T4 Mult-VAE 2 capacity", "task4", "mult_vae", {
        **task4_mult_vae_best_values,
        "hidden_dimension": [600, 1000],
        "latent_dimension": [200, 400],
    }, budget_minutes=10)

if run_task5:
    # Mult-VAE: dropout x KL weight (4 runs, 87k users x 50k books: the slowest Mult-VAE)
    task5_mult_vae_sweep = run_timeboxed_sweep("T5 Mult-VAE dropout / KL", "task5", "mult_vae", {
        "input_dropout": [0.3, 0.5],
        "kl_weight_max": [0.2, 0.5],
    }, budget_minutes=25)

    # SASRec stage 1 - width x dropout (4 runs; Kang & McAuley 2018: d = 50, dropout 0.5 on sparse data)
    task5_sasrec_sweep = run_timeboxed_sweep("T5 SASRec 1 width / dropout", "task5", "sasrec", {
        "hidden_dimension": [64, 128],
        "dropout": [0.2, 0.5],
    }, budget_minutes=28)
    task5_sasrec_best_values = best_values_of(task5_sasrec_sweep)

    # SASRec stage 2 - learning rate at the best width / dropout (2 runs; config 1e-3 is in stage 1)
    task5_sasrec_learning_rate_sweep = run_timeboxed_sweep("T5 SASRec 2 learning rate", "task5", "sasrec", {
        **task5_sasrec_best_values,
        "learning_rate": [5e-4, 2e-3],
    }, budget_minutes=14)

## Phase 4 - refinement queue (`overnight_queue.json`)
Runs every entry of the queue file it has not run yet (by `name`), re-reading the file every minute, until the overall end.

In [ ]:
def read_queue() -> dict:
    try:
        return json.loads(QUEUE_PATH.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as error:
        print(f"queue file not readable ({type(error).__name__}: {error}) -> retry in a minute", flush=True)
        return {"sweeps": []}


if run_queue:
    finished_queue_entries = set()
    while True:
        queue = read_queue()
        if queue.get("end_at"):                 # the queue file can move the overall end
            overall_end = datetime.strptime(queue["end_at"], "%Y-%m-%d %H:%M")
        if datetime.now() >= overall_end - timedelta(minutes=1):
            print(f"overall end {overall_end:%H:%M} reached -> queue stopped", flush=True)
            break
        pending_entries = [entry for entry in queue.get("sweeps", []) if entry["name"] not in finished_queue_entries]
        if not pending_entries:
            time.sleep(60)
            continue
        entry = pending_entries[0]
        finished_queue_entries.add(entry["name"])
        run_timeboxed_sweep(f"queue: {entry['name']}", entry["task"], entry["model"], entry["grid"],
                            entry.get("budget_minutes", 20), **entry.get("options", {}))

## Summary
One line per sweep of this notebook (also in `output/overnight_log.txt`), then every sweep file of the four tasks.

In [ ]:
for label, report in overnight_results.items():
    best = report["best"]
    best_text = f"{best['score']:.5f} with {best['swept_values']}" if best else "no finished run"
    print(f"{label:<42} {report['status']:<24} {report['runs_finished']:>9} | best {best_text}")

for task_name in ("task2", "task4", "task5", "task6"):
    print(f"\n---- {task_name} ----")
    print_sweep_overview(task_name)

keep_computer_awake(False)
write_log("overnight sweep finished")